# Stage 3 — Extraction Pipeline

**Project:** LLM-Based Information Extraction from Hospitality Request Emails: A Prompt Engineering Approach

**Author:** Mikail Imran Shaukat

---

## What this notebook does

This notebook runs the full LLM-based information extraction pipeline across all 9 experimental configurations.

For each of the 50 synthetic emails from Stage 2, the pipeline:
1. Sends the email to an LLM with a system prompt instructing it to extract 12 predefined fields and return them as a structured JSON object
2. Parses the JSON response and identifies any fields that were not found in the email (null values)
3. If any fields are missing, generates a draft follow-up email requesting only the missing information. Draft replies are only generated using GPT-4o in combination with Prompt 3.

This is repeated across a 3×3 experimental matrix, three prompt variants and three model variants producing 450 individual extraction results in total.

**Input:** `Stage 2 - Synthetic Data.csv` — the 50 synthetic emails from Stage 2  
**Output:** `Stage 3 - Extraction Results.csv` — 450 rows of extracted field values, missing fields, and draft replies.

---
## Cell 1 — Setup

Install and import the required libraries and initialise the OpenAI client.

The API key is stored securely in Colab secrets under the name `OPENAI_API_KEY` and is never hardcoded in the notebook.

In [1]:
!pip install openai -q

from google.colab import userdata
from openai import OpenAI
import json
import pandas as pd
import time

client = OpenAI(api_key=userdata.get('OPENAI_API_KEY'))
print('Client set up successfully')

Client set up successfully


---
## Cell 2 — Field Schema and Model Variants

The 12 extraction fields are defined here as a list. These are the fields co-defined with Xebia's hospitality staff in Stage 1 and form the basis of every prompt and every evaluation in this project.

The three model variants are defined as a dictionary. These represent three different capability and cost tiers:

- **GPT-4o** — flagship general-purpose model, used as the reference point
- **GPT-4o-mini** — cost-optimised variant, approximately 15–20× cheaper per token than GPT-4o
- **GPT-4.1-nano** — lowest cost model, optimised for structured extraction and low-latency tasks

In [2]:
FIELDS = [
    "event_date",
    "event_type",
    "event_time",
    "meal_type",
    "attendee_count",
    "external_guests",
    "catering_type",
    "office_location",
    "dietary_requirements",
    "room_location",
    "av_equipment",
    "extra_arrangements"
]

MODELS = {
    "gpt-4o":       "gpt-4o",
    "gpt-4o-mini":  "gpt-4o-mini",
    "gpt-4.1-nano": "gpt-4.1-nano",
}

print(f'Fields defined: {len(FIELDS)}')
print(f'Models defined: {list(MODELS.keys())}')

Fields defined: 12
Models defined: ['gpt-4o', 'gpt-4o-mini', 'gpt-4.1-nano']


---
## Cell 3 — Load Dataset

The 50 synthetic emails generated in Stage 2 are loaded here. The two assertions check that the file has the expected columns before anything else runs. This prevents errors later in the pipeline if the file is missing a column.

In [3]:
df = pd.read_csv('Stage 2 - Synthetic Data.csv')

assert 'email_id' in df.columns, "Dataset must have 'email_id' column"
assert 'email_text' in df.columns, "Dataset must have 'email_text' column"

print(f'Loaded {len(df)} emails')
print()
print(df.head())

Loaded 50 emails

   email_id                                         email_text
0         1  Hello Team,\n\nOn Thursday, March 5th, we will...
1         2  Hi Hospitality,\n\nI am organizing a tech talk...
2         3  Good morning,\n\nWe have a partner event sched...
3         4  Hi there,\n\nI need to book a space for a last...
4         5  Hi Team,\n\nI hope you’re doing well. I’m look...


---
## Prompt Design

Three prompt variants were designed to test the impact of increasing schema guidance on extraction performance. Each variant builds on the previous one by adding more context and instructions.

All three prompts share the same output rules:
- Return a raw JSON object with exactly the 12 field names as keys
- Return `null` for any field not mentioned in the email
- If the sender explicitly states they will confirm a value later (e.g. *"I will send dietary requirements by Thursday"*), extract the value as `"To be confirmed"` rather than null

The `"To be confirmed"` rule was introduced after the initial annotation of ground truth data, where it became clear that some emails explicitly deferred certain information rather than omitting it entirely.

---
### Cell 4 — Prompt 1: Zero-Shot Baseline

Prompt 1 provides the model with the least possible instructiosn. Just the 12 field names and the output rules. No definitions, no examples, no context about Xebia or the hospitality team.

This is known as **zero-shot prompting**: the model receives no examples and must rely entirely on its pre-trained knowledge to perform the task. The theory is that by being pre-trained on vast amounts of text, LLMs gain sufficient general knowledge of common business concepts to infer the meaning of field names such as `event_date` or `attendee_count` without additional guidance.

Prompt 1 serves as the experimental baseline. If this variant performs well, it indicates that LLMs are powerful enough for this type of domain-specific extraction task without any additional prompt engineering.

In [4]:
PROMPT_1 = """Extract the following 12 fields from the email

- event_date
- event_type
- event_time
- meal_type
- attendee_count
- external_guests
- catering_type
- office_location
- dietary_requirements
- room_location
- av_equipment
- extra_arrangements

Rules:
- Return ONLY a valid JSON object with exactly these 12 field names as keys
- If a field is not mentioned in the email, return null for that field
- If a sender explicitly states they will provide a value later, extract as "To be confirmed"
- Do not add any extra text, explanation or markdown. Just the raw JSON object"""

print('Prompt 1 defined successfully')

Prompt 1 defined successfully


---
### Cell 5 — Prompt 2: Schema-Guided Prompting

Prompt 2 builds on Prompt 1 by adding two components:

**1. A contextual introduction** — The model is told it is acting as an information extraction assistant for Xebia's hospitality back-office team. Framing the task as a real-world problem.

**2. Field definitions** — A natural language definition is provided for each of the 12 fields directly in the prompt.

Prompt 2 tests whether schema-level guidance produces higher performance scores over the zero-shot baseline.

In [5]:
PROMPT_2 = """You are an information extraction assistant for Xebia's hospitality back-office team.
Your task is to extract specific information from hospitality request emails.

Extract the following 12 fields from the email. Use the field definitions below to guide your extraction:

- event_date: The date the event needs to take place.
- event_type: The purpose of the event.
- event_time: Start and/or end time of the event.
- meal_type: The type of meal to be included, if any.
- attendee_count: Total number of people expected to attend.
- external_guests: Whether the event includes external guests or is limited to internal Xebia employees.
- catering_type: The type of catering required, given that a meal is included.
- office_location: The Xebia office at which the event will take place.
- dietary_requirements: Any dietary restrictions or requirements among attendees.
- room_location: The specific room or space within the office where the event is to be held.
- av_equipment: The type of audio-visual equipment required for the event.
- extra_arrangements: Any special or additional arrangements to be made for the event.

Rules:
- Return ONLY a valid JSON object with exactly these 12 field names as keys
- If a field is not mentioned in the email, return null for that field
- If a sender explicitly states they will provide a value later, extract as "To be confirmed"
- Do not add any extra text, explanation or markdown — just the raw JSON object"""

print('Prompt 2 defined successfully')

Prompt 2 defined successfully


---
### Cell 6 — Prompt 3: Few-Shot In-Context Learning

Prompt 3 builds on Prompt 2 by adding two real emails from the dataset alongside their manually annotated ground truth outputs as in-context examples.

This is known as **few-shot prompting**. Rather than relying solely on field definitions, the model is shown concrete examples of what the expected input and output looks like.

The two examples were selected from the real email dataset collected in Stage 1, not from the 50 synthetic emails used for evaluation. This is an important design choice because using evaluation emails as examples would mean the model has already seen those emails, which would invalidate the evaluation results. By taking examples only from the real emails, the evaluation remains a fair test of the model's extraction capability on unseen data.



In [6]:
EXAMPLE_1_EMAIL = """Morning,
On Wednesday 20 May, a team from Microsoft will come to us to organize a meeting. They come with a total of 20 people. I have already reserved
A201 for this. They will have lunch with the normal lunch, so good to take this into account.
Do you want to prepare the space for 26 people and then with only chairs in a large circle? Some Xebians join in the morning.
If there are any questions, please let me know."""

EXAMPLE_1_OUTPUT = """{
  "event_date": "Wednesday 20 May",
  "event_type": "Meeting with Microsoft",
  "event_time": null,
  "meal_type": "Lunch",
  "attendee_count": "26",
  "external_guests": "Microsoft team",
  "catering_type": "Normal lunch",
  "office_location": null,
  "dietary_requirements": null,
  "room_location": "A201",
  "av_equipment": null,
  "extra_arrangements": "Room setup with chairs in large circle"
}"""

EXAMPLE_2_EMAIL = """Morning,
On April 29, one of our customers will come to the office. Can you arrange a catered lunch for this and reserve a table for this? Preferably with
sandwiches and a soup.
Concerns a total of 10 people including our own colleagues. I have requested dietary requirements and will let you know no later than thursday.
Also, can you put a platter with goodies and put it in the bangalore together with some fruit, also arrange a snack for the afternoon."""

EXAMPLE_2_OUTPUT = """{
  "event_date": "April 29",
  "event_type": "Customer Visit",
  "event_time": null,
  "meal_type": "Lunch",
  "attendee_count": "10",
  "external_guests": "One of our customers",
  "catering_type": "Catered lunch with sandwiches and soup",
  "office_location": null,
  "dietary_requirements": "To be confirmed",
  "room_location": null,
  "av_equipment": null,
  "extra_arrangements": "Platter with goodies and fruit in Bangalore, snack in the afternoon"
}"""

PROMPT_3 = f"""You are an information extraction assistant for Xebia's hospitality back-office team.
Your task is to extract specific information from hospitality request emails.

Extract the following 12 fields from the email. Use the field definitions below to guide your extraction:

- event_date: The date the event needs to take place.
- event_type: The purpose of the event.
- event_time: Start and/or end time of the event.
- meal_type: The type of meal to be included, if any.
- attendee_count: Total number of people expected to attend.
- external_guests: Whether the event includes external guests or is limited to internal Xebia employees.
- catering_type: The type of catering required, given that a meal is included.
- office_location: The Xebia office at which the event will take place.
- dietary_requirements: Any dietary restrictions or requirements among attendees.
- room_location: The specific room or space within the office where the event is to be held.
- av_equipment: The type of audio-visual equipment required for the event.
- extra_arrangements: Any special or additional arrangements to be made for the event.

Rules:
- Return ONLY a valid JSON object with exactly these 12 field names as keys
- If a field is not mentioned in the email, return null for that field
- If a sender explicitly states they will provide a value later, extract as "To be confirmed"
- Do not add any extra text, explanation or markdown — just the raw JSON object

Below are two examples of correctly extracted emails:

---
Example 1:
Email: {EXAMPLE_1_EMAIL}
Output: {EXAMPLE_1_OUTPUT}

---
Example 2:
Email: {EXAMPLE_2_EMAIL}
Output: {EXAMPLE_2_OUTPUT}

---
Now extract from the email below using exactly the same format."""

print('Prompt 3 defined successfully')

Prompt 3 defined successfully


---
## Cell 7 — Bundle Prompts

All three prompts are collected into a single dictionary here. This makes it easy to loop over all three in the extraction run without repeating code.

In [7]:
PROMPTS = {
    "v1": PROMPT_1,
    "v2": PROMPT_2,
    "v3": PROMPT_3,
}

print('All prompts bundled successfully')
for name, prompt in PROMPTS.items():
    print(f'  {name}: {len(prompt)} characters')

All prompts bundled successfully
  v1: 570 characters
  v2: 1452 characters
  v3: 3377 characters


---
## Cell 8 — Pipeline Functions

Four functions are defined here that form the pipeline. Every extraction in the full run passes through these.

**`clean_json(raw)`**  
The OpenAI API occasionally wraps its response in markdown code fences (` ```json ``` `) despite being instructed not to. This function strips those fences before the response is parsed. Without this step, the JSON parser would crash because it does not recognise the backticks as valid JSON.

**`extract_fields(email_text, prompt, model)`**  
Sends a single email to the specified model using the specified prompt and returns the extracted fields as a Python dictionary. The prompt is passed as the system message (setting the overall instruction) and the email text is passed as the user message (the content to extract from). Temperature is set to 0 to ensure reproducible results across all runs.

**`get_missing_fields(extracted_fields)`**  
Takes the extracted dictionary and returns a list of any field names where the value is null. These are the fields that were not found in the email and will be requested in the follow-up reply.

**`generate_reply(email_text, missing_fields)`**  
If any fields are missing, this function generates a short professional follow-up email requesting only the missing information. It uses GPT-4o at temperature 0.8 to produce naturally varied replies. If no fields are missing, it returns a confirmation message and skips the API call entirely.

In [8]:
def clean_json(raw: str) -> str:
    cleaned = raw.strip()
    if cleaned.startswith('```'):
        lines = cleaned.split('\n')
        lines = [l for l in lines if not l.strip().startswith('```')]
        cleaned = '\n'.join(lines).strip()
    return cleaned

print('clean_json defined')


def extract_fields(email_text, prompt, model):
    response = client.chat.completions.create(
        model=model,
        messages=[
            {'role': 'system', 'content': prompt},
            {'role': 'user',   'content': f'Email:\n{email_text}'}
        ],
        temperature=0,
    )
    raw = response.choices[0].message.content
    cleaned = clean_json(raw)
    return json.loads(cleaned)

print('extract_fields defined')


def get_missing_fields(extracted_fields):
    missing = []
    for field, value in extracted_fields.items():
        if value is None:
            missing.append(field)
    return missing

print('get_missing_fields defined')


def generate_reply(email_text, missing_fields):
    if not missing_fields:
        return 'No follow-up needed — all fields extracted.'

    missing_list = '\n'.join(f'- {field.replace("_", " ")}' for field in missing_fields)

    reply_prompt = f"""You are a professional assistant working for Xebia's hospitality back-office team.

You received the following hospitality request email:
{email_text}

The following information is missing from this request and is needed to process it:
{missing_list}

Write a short, polite and professional follow-up email requesting only the missing information listed above. Do not ask for information that was already provided. Keep the tone friendly as it is internal office communication."""

    response = client.chat.completions.create(
        model='gpt-4o',
        messages=[{'role': 'user', 'content': reply_prompt}],
        temperature=0.8,
    )
    return response.choices[0].message.content

print('generate_reply defined')
print()
print('All functions ready')

clean_json defined
extract_fields defined
get_missing_fields defined
generate_reply defined

All functions ready


---
## Cell 9 — Test Run

Before running the full pipeline across all 450 combinations, a single email is tested across all 9 configurations here. This acted as a check to confirm that all three prompts and all three models are working correctly before committing to the full run.


In [9]:
test_email_id   = df.iloc[0]['email_id']
test_email_text = df.iloc[0]['email_text']

print(f'Testing with Email ID: {test_email_id}')
print(f'Email preview: {test_email_text[:100]}...')
print()

for prompt_name, prompt_text in PROMPTS.items():
    for model_name, model_id in MODELS.items():
        print(f'Testing {prompt_name} + {model_name}...', end=' ', flush=True)
        result = extract_fields(test_email_text, prompt_text, model_id)
        filled = sum(1 for v in result.values() if v is not None)
        print(f'✓ ({filled}/12 fields extracted)')

print()
print('All 9 configurations passed — ready for full run')

Testing with Email ID: 1
Email preview: Hello Team,

On Thursday, March 5th, we will be hosting a workshop for a group of potential clients....

Testing v1 + gpt-4o... ✓ (8/12 fields extracted)
Testing v1 + gpt-4o-mini... ✓ (8/12 fields extracted)
Testing v1 + gpt-4.1-nano... ✓ (8/12 fields extracted)
Testing v2 + gpt-4o... ✓ (9/12 fields extracted)
Testing v2 + gpt-4o-mini... ✓ (9/12 fields extracted)
Testing v2 + gpt-4.1-nano... ✓ (9/12 fields extracted)
Testing v3 + gpt-4o... ✓ (9/12 fields extracted)
Testing v3 + gpt-4o-mini... ✓ (9/12 fields extracted)
Testing v3 + gpt-4.1-nano... ✓ (9/12 fields extracted)

All 9 configurations passed — ready for full run


---
## Cell 10 — Full Extraction Run

This cell runs the complete extraction pipeline across all 450 combinations.

For each combination, the extracted fields and the list of missing fields are stored. Draft follow-up replies are generated only for the Prompt 3 + GPT-4o combination. This is because reply quality is not a part of the comparative evaluation. Running reply generation for all 9 configurations would add significant API cost and time without contributing to the evaluation results.

A progress counter is printed for each extraction so that the run is monitored.

In [ ]:
results = []
total   = len(df) * len(PROMPTS) * len(MODELS)
count   = 0

for prompt_name, prompt_text in PROMPTS.items():
    for model_name, model_id in MODELS.items():

        print(f'\n{"="*50}')
        print(f'Prompt={prompt_name} | Model={model_name}')
        print(f'{"="*50}')

        for _, row in df.iterrows():
            email_id   = row['email_id']
            email_text = row['email_text']
            count += 1

            print(f'  [{count}/{total}] Email {email_id}...', end=' ', flush=True)

            extracted = extract_fields(email_text, prompt_text, model_id)
            missing   = get_missing_fields(extracted)

            # Generate reply only for v3 + gpt-4o
            reply = None
            if prompt_name == 'v3' and model_name == 'gpt-4o':
                reply = generate_reply(email_text, missing)

            results.append({
                'prompt_version': prompt_name,
                'model_name':     model_name,
                'email_id':       email_id,
                **{f: extracted.get(f) for f in FIELDS},
                'missing_fields': json.dumps(missing),
                'draft_reply':    reply,
            })

            filled = sum(1 for v in extracted.values() if v is not None)
            print(f'✓ ({filled}/12 fields)')

print(f'\n{"="*50}')
print('EXTRACTION COMPLETE')
print(f'Total rows: {len(results)}')

---
## Cell 11 — Save Results

The results are saved to a CSV file here. Each row represents one email processed by one prompt-model configuration, giving 450 rows in total.

The summary printed below confirms that each of the 9 configurations processed exactly 50 emails, as expected.

In [ ]:
df_results = pd.DataFrame(results)
df_results.to_csv('Stage 3 - Extraction Results.csv', index=False)

print(f'Results saved to Stage 3 - Extraction Results.csv')
print(f'Shape: {df_results.shape}')
print()
print(df_results.groupby(['prompt_version', 'model_name']).size())